# GPT-2 on Different Tasks

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

# ------------------------
# Setup
# ------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model_name = "gpt2-medium"  # change to "gpt2" if system is weak
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)

def generate_text(prompt, max_length=150):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output = model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=0.9,
        top_k=50,
        top_p=0.95
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)



Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2-medium
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...23}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
def generate_text(prompt, max_length=150):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    output = model.generate(
        **inputs,
        max_length=max_length,
        do_sample=True,
        temperature=1.2,
        top_k=50,
        top_p=0.95
    )
    return tokenizer.decode(output[0], skip_special_tokens=True)

In [18]:
# ------------------------
# 1️⃣ Question Answering
# ------------------------
qa_prompt = """
Answer the following question clearly.

Question: What is the capital of Japan?
Answer:
"""

print("\n--- Question Answering ---")
print(generate_text(qa_prompt, 40))



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- Question Answering ---

Answer the following question clearly.

Question: What is the capital of Japan?
Answer:

Answer is the yen rate. Yen rates are set by the Central Bank. They differ


In [11]:
# ------------------------
# 2️⃣ Summarization
# ------------------------
sample_text = """
Artificial Intelligence is transforming industries worldwide. 
From healthcare to finance, AI systems help automate tasks, 
analyze massive datasets, and improve decision-making. 
However, ethical concerns such as bias, privacy, and job displacement remain important challenges.
"""

summ_prompt = f"""
Summarize the following text in 2-3 sentences.

Text:
{sample_text}

Summary:
"""

print("\n--- Summarization ---")
print(generate_text(summ_prompt, 120))



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- Summarization ---

Summarize the following text in 2-3 sentences.

Text:

Artificial Intelligence is transforming industries worldwide. 
From healthcare to finance, AI systems help automate tasks, 
analyze massive datasets, and improve decision-making. 
However, ethical concerns such as bias, privacy, and job displacement remain important challenges.


Summary:

Artificial Intelligence technology is being developed for a multitude of use cases and different contexts.

Artificial Intelligence is being utilized in all domains where human knowledge and skills are required,

including social, medical


In [10]:
# ------------------------
# 3️⃣ Sentiment Classification
# ------------------------
sent_prompt = """
Classify the sentiment as Positive, Negative, or Neutral.

Text: I absolutely loved the new smartphone. The performance is amazing.
Sentiment:
"""

print("\n--- Sentiment Classification ---")
print(generate_text(sent_prompt, 50))



Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- Sentiment Classification ---

Classify the sentiment as Positive, Negative, or Neutral.

Text: I absolutely loved the new smartphone. The performance is amazing.
Sentiment:

Sentiment: I feel like the new smartphone is an absolute steal. If


In [5]:

# ------------------------
# 4️⃣ Translation
# ------------------------
trans_prompt = """
Translate the following English sentence to French.

English: Machine learning is a fascinating field of study.
French:
"""

print("\n--- Translation ---")
print(generate_text(trans_prompt, 50))




Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- Translation ---

Translate the following English sentence to French.

English: Machine learning is a fascinating field of study.
French:

How to use the search box.

The search box has three places where you can enter the words that


In [3]:
# ------------------------
# 5️⃣ Creative Text Generation
# ------------------------
creative_prompt = """
Continue the story creatively.

Story:
In the year 2050, humans discovered a mysterious signal coming from deep space.
"""

print("\n--- Creative Writing ---")
print(generate_text(creative_prompt, 150))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



--- Creative Writing ---

Continue the story creatively.

Story:
In the year 2050, humans discovered a mysterious signal coming from deep space.

The signal was to be a new weapon that could destroy the universe in a matter of days.

The aliens had developed a new way of controlling the universe.

It was the only way they could stop humanity.

However, the aliens were very strong.

They had already established bases on Earth and Mars.

All they had to do was to find the signal and destroy it.

"You idiot! You are the one that created this weapon in the first place! You're the one that invented nuclear weapons!"

"A weapon that could destroy the universe in


# Inference and Findings

## Observations

After experimenting with GPT-2 (small and medium variants) across multiple NLP tasks, the following patterns were observed:

### 1. Stronger Performance In:
- Creative text generation  
- Story continuation  
- Open-ended summarization  

The model was able to generate coherent and grammatically correct text when the task aligned with its pretraining objective (next-token prediction).

### 2. Weak Performance In:
- Question Answering (fact-based)
- Sentiment classification
- Structured output tasks
- Precise instruction-following

The outputs were often:
- Incomplete
- Hallucinated
- Inconsistent
- Not aligned strictly with the task format

---

## Reason Behind This Behavior

GPT-2 was trained using **causal language modeling (next-token prediction)** on large-scale web text. It was NOT:

- Instruction-tuned
- Reinforcement learning aligned (RLHF)
- Fine-tuned for specific downstream tasks
- Optimized for structured reasoning or classification

Therefore:

- It performs well in **natural continuation tasks**, because generation is its native capability.
- It struggles in **task-specific formats**, because it was never explicitly trained to follow instructions or produce constrained outputs.

Additionally:

- GPT-2-medium (355M parameters) still has limited capacity compared to modern LLMs (7B–70B+).
- It lacks alignment training that improves factual consistency and task adherence.

---

## Key Findings

1. GPT-2 is fundamentally a **general language generator**, not a task-specialized model.
2. Prompt engineering alone cannot fully compensate for the lack of instruction tuning.
3. Modern LLMs outperform GPT-2 primarily due to:
   - Larger parameter scales
   - Instruction fine-tuning
   - Reinforcement learning alignment
   - Better curated training data

---

## Conclusion

While GPT-2-medium demonstrates reasonable performance in open-ended generation tasks, it is not suitable for real-world applications requiring:

- Reliable question answering
- Accurate classification
- Structured outputs
- Instruction compliance

This experiment highlights the evolution of language models from pure next-token predictors (GPT-2) to instruction-aligned, large-scale foundation models used today.

---